# NBA Oracle — Colab T4 multi-model pipeline (v3, stratified-month verified)Reproduces the 2026-04-28 overnight result that promoted TabICL-186 at walk-forward holdout Brier 0.21139, **plus a stratified-by-month sanity check** so we can tell whether 0.21139 was a real generalization win or just a lucky last-15%-of-rows window.Pipeline:1. Pull 9 seasons + 7 enriched feeds + engine.py from HF dataset.2. Run `engine.build()` end-to-end → ~11,500 games × ~7,246 cols (~4,500+ alive).3. Walk-forward holdout = last 15% by row order. CPCV-10 on the train portion.4. Score 3 models on the same train/holdout: XGBoost (alive), LightGBM (alive), TabICL (top-186 by variance, ctx∈{2048,3072}, temp∈{1.0,1.08}).5. Pick winner by **walk-forward holdout Brier** (not CV — the corpus shows ~−0.01 negative gap on every model, so holdout is the honest metric).6. **Stratified-by-month sanity check** on the winner: split last 15% of *months* as holdout, recompute Brier. If it's >0.005 worse than walk-forward holdout, the win is window-lucky and we revert hype.7. Isotonic-calibrate (5-fold OOF) and report calibrated holdout Brier as a third estimator.8. Archive pkl, promote to `LBJLincoln26/nba-oracle-model` only if walk-forward holdout < current production holdout.Setup: Runtime → GPU T4. Tools → Secrets → `HF_TOKEN` (LBJLincoln26 owner). Run All. ~30–45 min.

In [ ]:
# Cell 1 — install + token!pip install -q tabicl xgboost lightgbm huggingface_hub scikit-learn==1.6.1 2>&1 | tail -3import os, sys, json, time, pickle, warningswarnings.filterwarnings('ignore')from datetime import datetime, timezonefrom pathlib import Pathfrom google.colab import userdataHF_TOKEN = userdata.get('HF_TOKEN')os.environ['HF_TOKEN'] = HF_TOKENprint('token len=', len(HF_TOKEN))

In [ ]:
# Cell 2 — pull all data (9 seasons + 7 enriched feeds + engine.py)from huggingface_hub import hf_hub_downloadDATASET = 'LBJLincoln26/nba-feature-cache'DATA_DIR = Path('/content/feature_data')DATA_DIR.mkdir(exist_ok=True)DATA_FILES = ['engine.py','referee_data.json','player_data_merged.json',              'quarter_data.json','polymarket_data.json','full-odds-2025-26.json',              'tracking_data.json']SEASON_FILES = [f'games-{y}.json' for y in                ['2017-18','2018-19','2019-20','2020-21','2021-22','2022-23','2023-24','2024-25','2025-26']]for f in DATA_FILES + SEASON_FILES:    p = hf_hub_download(repo_id=DATASET, filename=f'feature_data/{f}', repo_type='dataset', token=HF_TOKEN)    target = DATA_DIR / f    if target.exists() or target.is_symlink():        target.unlink()    target.symlink_to(p)    print(f'  {f:35s} {Path(p).stat().st_size/1024/1024:6.2f} MB')print(f'\n{len(DATA_FILES)} data + {len(SEASON_FILES)} season files ready')

In [ ]:
# Cell 3 — load engine + concat seasons + wire all feedsimport importlib.utilspec = importlib.util.spec_from_file_location('engine', str(DATA_DIR / 'engine.py'))engine_mod = importlib.util.module_from_spec(spec)sys.modules['engine'] = engine_modspec.loader.exec_module(engine_mod)print('engine_version:', engine_mod.ENGINE_VERSION)games = []for season_file in SEASON_FILES:    raw = json.loads((DATA_DIR / season_file).read_text())    season_games = raw.get('games', raw) if isinstance(raw, dict) else raw    games.extend(season_games)    print(f'  {season_file}: {len(season_games)} games')print(f'  TOTAL: {len(games)} games')referee_data = json.loads((DATA_DIR / 'referee_data.json').read_text())tracking_data = json.loads((DATA_DIR / 'tracking_data.json').read_text())raw_pd = json.loads((DATA_DIR / 'player_data_merged.json').read_text())player_data = {}for k, v in raw_pd.items():    if '|' in k:        team, date = k.split('|', 1)        player_data[(team, date)] = v    else:        player_data[k] = vraw_qd = json.loads((DATA_DIR / 'quarter_data.json').read_text())quarter_data = {tuple(k.split('|',1)): v for k, v in raw_qd.items()}raw_pm = json.loads((DATA_DIR / 'polymarket_data.json').read_text())market_data = {}for g in games:    gid = g.get('game_id', '')    d = (g.get('game_date') or '')[:10]    h = (g.get('home',{}) or {}).get('team_abbr','') if isinstance(g.get('home'), dict) else ''    a = (g.get('away',{}) or {}).get('team_abbr','') if isinstance(g.get('away'), dict) else ''    key = f'{d}|{h}|{a}'    if key in raw_pm:        market_data[gid] = raw_pm[key]raw_odds = json.loads((DATA_DIR / 'full-odds-2025-26.json').read_text())odds_data = {}for k, v in raw_odds.items():    if not isinstance(v, dict) or '_' not in k or '@' not in k:        continue    try:        date_part, matchup = k.split('_', 1)        away, home = matchup.split('@', 1)    except ValueError:        continue    base = v.get('base', {})    odds_data[(date_part[:10], home, away)] = dict(base)print(f'\nfeeds: ref={len(referee_data)} player={len(player_data)} quarter={len(quarter_data)} market={len(market_data)} odds={len(odds_data)} tracking={len(tracking_data)}')assert len(referee_data) > 100, 'referee_data empty'assert len(player_data) > 100, 'player_data empty'assert len(odds_data) > 100, 'odds_data empty (parser bug?)'

In [ ]:
# Cell 4 — build feature matrix on all gamesimport numpy as npengine = engine_mod.NBAFeatureEngine()t0 = time.time()X, y, feature_names = engine.build(    games,    referee_data=referee_data, player_data=player_data,    quarter_data=quarter_data, market_data=market_data,    odds_data=odds_data, tracking_data=tracking_data,)elapsed = time.time() - t0X = np.nan_to_num(np.array(X, dtype=np.float32))y = np.array(y, dtype=np.float64)print(f'build: {X.shape[0]} games x {X.shape[1]} cols in {elapsed:.0f}s; y_mean={y.mean():.3f}')# Capture game_dates aligned to X rows (for stratified-by-month sanity check).# engine.build() preserves game order, so the i-th row of X corresponds to games[i]# IF engine drops nothing — but since ENGINE drops games with missing labels we# need a fallback. Try to read engine.last_game_dates if exposed; otherwise fall# back to row-position-based month buckets in cell 9.game_dates = Nonetry:    game_dates = list(getattr(engine, 'last_game_dates', None) or [])    if len(game_dates) != X.shape[0]:        game_dates = Noneexcept Exception:    game_dates = Noneif game_dates is None:    # Best-effort: assume engine kept games in order with no drops    if len(games) == X.shape[0]:        game_dates = [(g.get('game_date') or '')[:10] for g in games]        print(f'game_dates aligned by row order ({len(game_dates)} entries)')    else:        print(f'WARN: engine kept {X.shape[0]} of {len(games)} games — month-stratified holdout will fall back to row-position buckets')variances = X.var(axis=0)alive_mask = variances > 1e-10alive_idx = np.where(alive_mask)[0]print(f'alive: {len(alive_idx)}/{X.shape[1]} ({len(alive_idx)/X.shape[1]*100:.1f}%)')X_full = X[:, alive_idx].astype(np.float32)ranked = sorted(alive_idx, key=lambda i: -variances[i])top186_idx = ranked[:186]X_186 = X[:, top186_idx].astype(np.float32)print(f'X_full (XGB/LGBM): {X_full.shape}    X_186 (TabICL): {X_186.shape}')assert X.shape[0] > 1000, f'too few games — only {X.shape[0]} survived engine filters'

In [ ]:
# Cell 5 — walk-forward holdout (last 15% by row order) + CPCV-10 setupfrom sklearn.metrics import brier_score_lossRANDOM_STATE = 1337N = len(X_full)HOLDOUT_FRAC = 0.15HOLDOUT_START = int(N * (1 - HOLDOUT_FRAC))X_train_full = X_full[:HOLDOUT_START]y_train = y[:HOLDOUT_START]X_holdout_full = X_full[HOLDOUT_START:]y_holdout = y[HOLDOUT_START:]X_train_186 = X_186[:HOLDOUT_START]X_holdout_186 = X_186[HOLDOUT_START:]print(f'TRAIN: {X_train_full.shape}    HOLDOUT (last {HOLDOUT_FRAC*100:.0f}%): {X_holdout_full.shape}')print(f'  train y_mean={y_train.mean():.3f}  holdout y_mean={y_holdout.mean():.3f}')N_FOLDS = 10EMBARGO = max(1, int(len(X_train_full) * 0.02))FOLD_SIZE = len(X_train_full) // N_FOLDSdef cpcv_folds(n):    for k in range(N_FOLDS):        lo = k * FOLD_SIZE        hi = lo + FOLD_SIZE if k < N_FOLDS - 1 else n        m = np.ones(n, dtype=bool)        m[max(0, lo - EMBARGO):min(n, hi + EMBARGO)] = False        yield np.where(m)[0], np.arange(lo, hi)all_results = []

In [ ]:
# Cell 6 — XGBoost: CPCV CV on train + walk-forward holdoutimport xgboost as xgbfold_briers = []oof = np.zeros(len(X_train_full))t0 = time.time()for tr, te in cpcv_folds(len(X_train_full)):    m = xgb.XGBClassifier(        n_estimators=500, max_depth=6, learning_rate=0.05,        subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,        random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0,    )    m.fit(X_train_full[tr], y_train[tr])    p = m.predict_proba(X_train_full[te])[:, 1]    oof[te] = p    fold_briers.append(float(brier_score_loss(y_train[te], p)))xgb_cv = float(np.mean(fold_briers))m = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,                     subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                     random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)m.fit(X_train_full, y_train)p_holdout = m.predict_proba(X_holdout_full)[:, 1]xgb_holdout = float(brier_score_loss(y_holdout, p_holdout))print(f'XGBoost: CV={xgb_cv:.5f} | walk-forward holdout={xgb_holdout:.5f} | gap={xgb_holdout-xgb_cv:+.5f} ({time.time()-t0:.0f}s)')all_results.append({'model':'xgboost','features':X_full.shape[1],                    'brier_cv':xgb_cv,'brier_holdout':xgb_holdout,                    'fold_briers':fold_briers,'oof':oof.tolist(),                    'holdout_preds':p_holdout.tolist()})

In [ ]:
# Cell 7 — LightGBM: same protocolimport lightgbm as lgbfold_briers = []oof = np.zeros(len(X_train_full))t0 = time.time()for tr, te in cpcv_folds(len(X_train_full)):    m = lgb.LGBMClassifier(        n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,        subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,        random_state=RANDOM_STATE, verbose=-1, force_col_wise=True,    )    m.fit(X_train_full[tr], y_train[tr])    p = m.predict_proba(X_train_full[te])[:, 1]    oof[te] = p    fold_briers.append(float(brier_score_loss(y_train[te], p)))lgb_cv = float(np.mean(fold_briers))m = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,                       subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                       random_state=RANDOM_STATE, verbose=-1, force_col_wise=True)m.fit(X_train_full, y_train)p_holdout = m.predict_proba(X_holdout_full)[:, 1]lgb_holdout = float(brier_score_loss(y_holdout, p_holdout))print(f'LightGBM: CV={lgb_cv:.5f} | walk-forward holdout={lgb_holdout:.5f} | gap={lgb_holdout-lgb_cv:+.5f} ({time.time()-t0:.0f}s)')all_results.append({'model':'lightgbm','features':X_full.shape[1],                    'brier_cv':lgb_cv,'brier_holdout':lgb_holdout,                    'fold_briers':fold_briers,'oof':oof.tolist(),                    'holdout_preds':p_holdout.tolist()})

In [ ]:
# Cell 8 — TabICL on top-186, sweep ctx,tempfrom tabicl import TabICLClassifierbest_tabicl = Nonefor ctx, temp in [(2048, 1.0), (3072, 1.0), (2048, 1.08)]:    fold_briers = []    oof = np.zeros(len(X_train_186))    t0 = time.time()    for tr, te in cpcv_folds(len(X_train_186)):        m = TabICLClassifier(n_estimators=1, softmax_temperature=temp, random_state=RANDOM_STATE)        sub_tr = tr[-ctx:]        m.fit(X_train_186[sub_tr], y_train[sub_tr])        p = m.predict_proba(X_train_186[te])[:, 1]        oof[te] = p        fold_briers.append(float(brier_score_loss(y_train[te], p)))    cv = float(np.mean(fold_briers))    print(f'TabICL ctx={ctx} temp={temp}: CV={cv:.5f} in {time.time()-t0:.0f}s')    if best_tabicl is None or cv < best_tabicl['brier_cv']:        best_tabicl = {'model':'tabicl','features':186,'brier_cv':cv,'ctx':ctx,'temp':temp,                       'fold_briers':fold_briers,'oof':oof.tolist()}m = TabICLClassifier(n_estimators=1, softmax_temperature=best_tabicl['temp'], random_state=RANDOM_STATE)m.fit(X_train_186[-best_tabicl['ctx']:], y_train[-best_tabicl['ctx']:])p_holdout = m.predict_proba(X_holdout_186)[:, 1]best_tabicl['brier_holdout'] = float(brier_score_loss(y_holdout, p_holdout))best_tabicl['holdout_preds'] = p_holdout.tolist()print(f"best TabICL: ctx={best_tabicl['ctx']} temp={best_tabicl['temp']} CV={best_tabicl['brier_cv']:.5f} holdout={best_tabicl['brier_holdout']:.5f}")all_results.append(best_tabicl)

In [ ]:
# Cell 9 — pick winner by walk-forward holdout, then STRATIFIED-BY-MONTH sanity checkfrom sklearn.isotonic import IsotonicRegressionwinner = min(all_results, key=lambda r: r['brier_holdout'])print(f"\n{'='*60}")print(f"WINNER (by walk-forward holdout): {winner['model']} on {winner['features']} features")print(f"  CV brier:      {winner['brier_cv']:.5f}")print(f"  Holdout brier: {winner['brier_holdout']:.5f}")print(f"{'='*60}")for r in all_results:    gap = r['brier_holdout'] - r['brier_cv']    flag = ' (overfit?)' if gap > 0.020 else ''    print(f"  {r['model']:10s} {r['features']:>5d} feats | CV {r['brier_cv']:.5f} | holdout {r['brier_holdout']:.5f} | gap {gap:+.5f}{flag}")# Stratified-by-month sanity check on the winner.# Splits last ~15% of MONTHS as holdout (no labels touched). If month-holdout# brier is >0.005 worse than walk-forward holdout brier, the win was window-lucky.print('\n--- stratified-by-month sanity check on winner ---')month_holdout_brier = Nonemonth_holdout_n = Nonen_ho_months = Noneif game_dates is not None and len(game_dates) == X.shape[0]:    months = np.array([(d or '')[:7] for d in game_dates])    unique_months = sorted(set(m for m in months if m))    n_ho_months = max(1, int(len(unique_months) * HOLDOUT_FRAC))    ho_months = set(unique_months[-n_ho_months:])    ho_mask = np.array([m in ho_months for m in months])    tr_mask = ~ho_mask    print(f'months total={len(unique_months)} holdout_months={n_ho_months} holdout_n={ho_mask.sum()} train_n={tr_mask.sum()}')    month_holdout_n = int(ho_mask.sum())    if month_holdout_n < 100:        print(f'WARN: month-holdout has only {month_holdout_n} games — skipping')    elif winner['model'] == 'tabicl':        mw = TabICLClassifier(n_estimators=1, softmax_temperature=winner['temp'], random_state=RANDOM_STATE)        tr_pos = np.where(tr_mask)[0][-winner['ctx']:]        mw.fit(X_186[tr_pos], y[tr_pos])        p = mw.predict_proba(X_186[ho_mask])[:, 1]        month_holdout_brier = float(brier_score_loss(y[ho_mask], p))    elif winner['model'] == 'xgboost':        mw = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,                               subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                               random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)        mw.fit(X_full[tr_mask], y[tr_mask])        p = mw.predict_proba(X_full[ho_mask])[:, 1]        month_holdout_brier = float(brier_score_loss(y[ho_mask], p))    else:        mw = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,                                subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                                random_state=RANDOM_STATE, verbose=-1, force_col_wise=True)        mw.fit(X_full[tr_mask], y[tr_mask])        p = mw.predict_proba(X_full[ho_mask])[:, 1]        month_holdout_brier = float(brier_score_loss(y[ho_mask], p))    if month_holdout_brier is not None:        delta = month_holdout_brier - winner['brier_holdout']        verdict = 'CONFIRMS' if delta <= 0.005 else 'CONTRADICTS (window-lucky)'        print(f'walk-forward holdout: {winner["brier_holdout"]:.5f}')        print(f'month-stratified holdout: {month_holdout_brier:.5f} (n={month_holdout_n}, {n_ho_months} months)')        print(f'delta = {delta:+.5f} -> {verdict}')else:    print('SKIPPED: game_dates not aligned to X rows — cannot bucket by month')

In [ ]:
# Cell 10 — isotonic + retrain winner on ALL data (incl. holdout) for production# isotonic uses TRAIN OOF only (clean) — applied to holdout preds to estimate# calibrated_holdout_brier as a third honest metric.oof = np.array(winner['oof'])iso = IsotonicRegression(out_of_bounds='clip').fit(oof, y_train)raw_b = brier_score_loss(y_train, oof)cal_b = brier_score_loss(y_train, iso.predict(oof))calibrated_holdout_brier = float(brier_score_loss(y_holdout, iso.predict(np.array(winner['holdout_preds']))))print(f'isotonic on train OOF: raw={raw_b:.5f} -> calibrated={cal_b:.5f}')print(f'isotonic applied to holdout preds: holdout_brier_calibrated={calibrated_holdout_brier:.5f}')if winner['model'] == 'xgboost':    final = xgb.XGBClassifier(n_estimators=500, max_depth=6, learning_rate=0.05,                              subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                              random_state=RANDOM_STATE, tree_method='hist', device='cuda', verbosity=0)    final.fit(X_full, y)    final_features = [feature_names[i] for i in alive_idx]    final_indices = list(map(int, alive_idx))elif winner['model'] == 'lightgbm':    final = lgb.LGBMClassifier(n_estimators=500, max_depth=8, num_leaves=63, learning_rate=0.05,                               subsample=0.8, colsample_bytree=0.6, reg_alpha=0.1, reg_lambda=1.0,                               random_state=RANDOM_STATE, verbose=-1, force_col_wise=True)    final.fit(X_full, y)    final_features = [feature_names[i] for i in alive_idx]    final_indices = list(map(int, alive_idx))else:    final = TabICLClassifier(n_estimators=1, softmax_temperature=winner['temp'], random_state=RANDOM_STATE)    final.fit(X_186[-winner['ctx']:], y[-winner['ctx']:])    final_features = [feature_names[i] for i in top186_idx]    final_indices = list(map(int, top186_idx))print(f'final {winner["model"]} retrained on all {len(y)} games × {len(final_features)} features')

In [ ]:
# Cell 11 — archive + auto-promote (gate uses walk-forward holdout, not raw CV)from huggingface_hub import HfApi, hf_hub_downloadutc = datetime.now(timezone.utc).strftime('%Y-%m-%dT%H-%M-%SZ')bundle = {    'model': final, 'calibrator': iso,    'feature_indices': final_indices, 'feature_names': final_features,    # cv_brier_mean is the HONEST CV (not overwritten by holdout). Keep both fields.    'cv_brier_mean': winner['brier_cv'],    'cv_brier_per_fold': winner.get('fold_briers', []),    'holdout_brier': winner['brier_holdout'],    'holdout_brier_calibrated': calibrated_holdout_brier,    'month_stratified_holdout_brier': month_holdout_brier,    'month_stratified_holdout_n': month_holdout_n,    'month_stratified_holdout_months': n_ho_months,    'config': {        'model_type': winner['model'], 'n_features': len(final_features),        'random_state': RANDOM_STATE, 'cpcv_folds': N_FOLDS,        'engine_version': engine_mod.ENGINE_VERSION,        'features_alive_total': int(len(alive_idx)),        'features_engine_total': int(X.shape[1]),        'all_results_summary': [{'model': r['model'], 'features': r['features'],                                  'brier_cv': r['brier_cv'], 'brier_holdout': r['brier_holdout']}                                 for r in all_results],        'walk_forward_holdout_frac': HOLDOUT_FRAC,        'holdout_n_games': len(y_holdout),    },    'n_samples': int(X.shape[0]),    'trained_at': utc, 'trained_on': f'colab-multi-{winner["model"]}',}if winner['model'] == 'tabicl':    bundle['config']['ctx_size'] = winner['ctx']    bundle['config']['softmax_temperature'] = winner['temp']PKL = f'/tmp/oracle-{winner["model"]}-{utc}.pkl'with open(PKL, 'wb') as f:    pickle.dump(bundle, f)print(f'pkl: {PKL} ({Path(PKL).stat().st_size/1024/1024:.1f} MB)')api = HfApi(token=HF_TOKEN)api.create_repo('LBJLincoln26/nba-oracle-archive', repo_type='dataset', private=False, exist_ok=True)api.upload_file(    path_or_fileobj=PKL,    path_in_repo=f'colab-multi-{winner["model"]}-{utc}.pkl',    repo_id='LBJLincoln26/nba-oracle-archive', repo_type='dataset',    commit_message=f'[colab-multi] {winner["model"]} CV {winner["brier_cv"]:.5f} holdout {winner["brier_holdout"]:.5f} cal {calibrated_holdout_brier:.5f} alive={len(alive_idx)}/{X.shape[1]}',)print('archived')# Promotion gate compares HOLDOUT vs production HOLDOUT (with backward-compat# fallback to old summaries that stored holdout in cv_brier_mean).try:    cur = json.load(open(hf_hub_download(        repo_id='LBJLincoln26/nba-oracle-model', filename='summary.json',        repo_type='dataset', token=HF_TOKEN)))    cur_holdout = float(cur.get('holdout_brier', cur.get('cv_brier_mean', 0.99)))except Exception:    cur_holdout = 0.99print(f'production holdout: {cur_holdout:.5f}')if winner['brier_holdout'] < cur_holdout:    api.upload_file(        path_or_fileobj=PKL, path_in_repo='nba-oracle.pkl',        repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset',        commit_message=f'[PROMOTE] {winner["model"]} holdout {winner["brier_holdout"]:.5f} (was {cur_holdout:.5f}, CV {winner["brier_cv"]:.5f}, cal_holdout {calibrated_holdout_brier:.5f})',    )    summary = {k: v for k, v in bundle.items() if k not in ('model', 'calibrator')}    summary['promoted_from_holdout'] = cur_holdout    api.upload_file(        path_or_fileobj=json.dumps(summary, indent=2, default=str).encode(),        path_in_repo='summary.json',        repo_id='LBJLincoln26/nba-oracle-model', repo_type='dataset',        commit_message=f'[PROMOTE] summary update — holdout {winner["brier_holdout"]:.5f} cal {calibrated_holdout_brier:.5f}',    )    print(f'\n*** PROMOTED: {winner["model"]} holdout {winner["brier_holdout"]:.5f} (was {cur_holdout:.5f}) ***')else:    print(f'NOT promoted: {winner["brier_holdout"]:.5f} >= {cur_holdout:.5f}')print('\n=== DONE ===')